# English to Spanish Translator

## Colab Training Notebook

This notebook trains and evaluates the Transformer model in Google Colab.

Before running the notebook:
- switch Colab to a GPU runtime
- review the training settings before starting
- expect the OPUS download and preprocessing stages to take time on the full corpus mix


## Mount the Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Clone Repository

This notebook can be uploaded to Colab by itself. The first code cell clones the project repository into `/content` if the project files are not already present.


In [2]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mathew-felix/english-spanish-translator.git"
REPO_DIR = Path("/content/english-spanish-translator")

def looks_like_repo(path: Path) -> bool:
    return (
        path.exists()
        and (path / ".git").exists()
        and (path / "source").exists()
        and (path / "run.py").exists()
    )

current_dir = Path.cwd()

if looks_like_repo(current_dir):
    project_dir = current_dir
elif looks_like_repo(REPO_DIR):
    project_dir = REPO_DIR
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    project_dir = REPO_DIR

os.chdir(project_dir)
print("Working directory:", Path.cwd())


Working directory: /content/english-spanish-translator


## 2. Install Dependencies

Colab already includes a CUDA-enabled PyTorch build. This cell installs the remaining project dependencies without replacing the Colab PyTorch packages.


In [3]:
from pathlib import Path

skip_packages = {"torch", "torchvision", "torchaudio", "typing_extensions", "typing-extensions", "numpy", "pandas"}
filtered_lines = []

for line in Path("requirements.txt").read_text().splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    package_name = stripped.split("==")[0].lower()
    if package_name in skip_packages:
        continue
    filtered_lines.append(stripped)

# Ensure typing_extensions isn't downgraded by other dependencies
filtered_lines.append("typing_extensions>=4.10.0")

Path("/tmp/colab-requirements.txt").write_text("\n".join(filtered_lines) + "\n")

!pip install -q -r /tmp/colab-requirements.txt

import sys
import importlib
if "typing_extensions" in sys.modules:
    importlib.reload(sys.modules["typing_extensions"])

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


## 3. Optional W&B Configuration

Set your Weights & Biases API key here if you want live cloud tracking. If the key is left blank, training will still run and log locally in offline mode.


In [4]:
!pip install -q --upgrade wandb
import os

# Clear the invalid key to allow training to proceed
WANDB_API_KEY = "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"
WANDB_PROJECT = "english-spanish-translator"
WANDB_ENTITY = ""

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
elif "WANDB_API_KEY" in os.environ:
    del os.environ["WANDB_API_KEY"]

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if os.getenv("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "online"
    print("W&B online logging enabled.")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B API key not set. Training will log offline in this Colab session.")

W&B online logging enabled.


## 4. Training Configuration

Set `QUICK_SANITY_RUN = True` for the first Colab pass. That mode downloads a smaller corpus subset, builds a small sampled train/test split, and runs a short training job to confirm the notebook works end to end.

For the full run on your RTX 6000 Pro setup, keep `QUICK_SANITY_RUN = False`. `MAX_SEQ_LENGTH=60` matches your stronger checkpoint setting, and `LEARNING_RATE=3e-4` is the safer full-run starting point.


In [5]:
QUICK_SANITY_RUN = False

# Full training defaults
NUM_EPOCHS = 30
BATCH_SIZE = 640
MAX_SEQ_LENGTH = 60
LEARNING_RATE = 4.5e-4
PATIENCE = 5
WARMUP_STEPS = 4000
BEAM_WIDTH = 4
BLEU_EVAL_BATCHES = 5
OPEN_SUBTITLES_MAX_ROWS = 2_000_000
NUM_WORKERS = 4

# Sanity-run overrides
SANITY_CORPORA = ("Europarl", "News-Commentary", "TED2020")
SANITY_SAMPLE_FRACTION = 0.01
SANITY_NUM_EPOCHS = 1
SANITY_BATCH_SIZE = 64
SANITY_LEARNING_RATE = 2e-4
SANITY_BLEU_EVAL_BATCHES = 1

if QUICK_SANITY_RUN:
    DOWNLOAD_CORPORA = SANITY_CORPORA
    EFFECTIVE_NUM_EPOCHS = SANITY_NUM_EPOCHS
    EFFECTIVE_BATCH_SIZE = SANITY_BATCH_SIZE
    EFFECTIVE_LEARNING_RATE = SANITY_LEARNING_RATE
    EFFECTIVE_BLEU_EVAL_BATCHES = SANITY_BLEU_EVAL_BATCHES
    EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS = 0
else:
    DOWNLOAD_CORPORA = ("Europarl", "News-Commentary", "TED2020", "OpenSubtitles")
    EFFECTIVE_NUM_EPOCHS = NUM_EPOCHS
    EFFECTIVE_BATCH_SIZE = BATCH_SIZE
    EFFECTIVE_LEARNING_RATE = LEARNING_RATE
    EFFECTIVE_BLEU_EVAL_BATCHES = BLEU_EVAL_BATCHES
    EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS = OPEN_SUBTITLES_MAX_ROWS

print("Quick sanity run:", QUICK_SANITY_RUN)
print("Corpora:", DOWNLOAD_CORPORA)
print("Epochs:", EFFECTIVE_NUM_EPOCHS)
print("Batch size:", EFFECTIVE_BATCH_SIZE)
print("Learning rate:", EFFECTIVE_LEARNING_RATE)
print("OpenSubtitles max rows:", EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS)


Quick sanity run: False
Corpora: ('Europarl', 'News-Commentary', 'TED2020', 'OpenSubtitles')
Epochs: 30
Batch size: 640
Learning rate: 0.00045
OpenSubtitles max rows: 2000000


## 5. Download Dataset

This cell downloads the English-Spanish corpus mix from OPUS. In quick sanity mode it skips OpenSubtitles so the first Colab pass stays manageable.


In [6]:
import source.DatasetDownload as dataset_download

dataset_download.OPUS_CORPORA = tuple(DOWNLOAD_CORPORA)
dataset_download.datasetDownload()

!du -sh data
!find data -maxdepth 2 -type f | head -n 20

Preparing Europarl (v8) with 2009073 aligned pairs.
Using existing archive './data/raw/Europarl.zip'.
Using existing extracted files in './data/raw/Europarl'.
Preparing News-Commentary (v16) with 49089 aligned pairs.
Using existing archive './data/raw/News-Commentary.zip'.
Using existing extracted files in './data/raw/News-Commentary'.
Preparing TED2020 (v1) with 416846 aligned pairs.
Using existing archive './data/raw/TED2020.zip'.
Using existing extracted files in './data/raw/TED2020'.
Preparing OpenSubtitles (v2024) with 105482431 aligned pairs.
Using existing archive './data/raw/OpenSubtitles.zip'.
Using existing extracted files in './data/raw/OpenSubtitles'.
11G	data
data/raw/OpenSubtitles.zip
data/raw/News-Commentary.zip
data/raw/TED2020.zip
data/raw/Europarl.zip


## 6. Preprocess Dataset

This cell merges the selected corpora into one bilingual CSV, filters noisy sentence pairs, and writes `train.csv` and `test.csv`. In quick sanity mode it also samples a small fraction of the merged corpus before the split.


In [7]:
import pandas as pd
import source.DatasetPreprocessing as dataset_preprocessing

if hasattr(dataset_preprocessing, 'CORPUS_LIMITS'):
    dataset_preprocessing.CORPUS_LIMITS = {
        corpus: dataset_preprocessing.CORPUS_LIMITS[corpus]
        for corpus in DOWNLOAD_CORPORA
        if corpus in dataset_preprocessing.CORPUS_LIMITS
    }

    if "OpenSubtitles" in dataset_preprocessing.CORPUS_LIMITS:
        dataset_preprocessing.OPEN_SUBTITLES_MAX_ROWS = EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS
        dataset_preprocessing.CORPUS_LIMITS["OpenSubtitles"] = EFFECTIVE_OPEN_SUBTITLES_MAX_ROWS

dataset_preprocessing.DatasetPreprocessing()

if QUICK_SANITY_RUN:
    sanity_path = "data/sanity_dataset.csv"
    dataset_preprocessing.SmallDataset(
        dataset_preprocessing.MERGED_DATASET_PATH,
        sanity_path,
        SANITY_SAMPLE_FRACTION,
    )
    dataset_preprocessing.Split_data(sanity_path)

for csv_path in ["data/english_spanish.csv", "data/train.csv", "data/test.csv"]:
    try:
        df = pd.read_csv(csv_path, nrows=5)
        print(f"\n{csv_path}")
        print(df.head())
    except FileNotFoundError:
        print(f"\n{csv_path} not found.")

!ls -lh data

Europarl: kept 1940734 pairs from 2009073 processed rows.
News-Commentary: kept 46904 pairs from 49089 processed rows.
TED2020: kept 403752 pairs from 416846 processed rows.
OpenSubtitles: kept 2000000 pairs from 2293562 processed rows.
Merged dataset written to './data/english_spanish.csv'.
Loaded inspection sample with 50000 rows and 3 columns from './data/english_spanish.csv'.

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   English  50000 non-null  object
 1   Spanish  50000 non-null  object
 2   Corpus   50000 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB
None

Missing Values:
English    0
Spanish    0
Corpus     0
dtype: int64

Duplicate Rows in sample: 0

Unique characters in English column: [' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 

## 7. Train Model

This cell keeps the project model architecture unchanged and only overrides the runtime training settings for Colab. In quick sanity mode it uses the lighter overrides defined above.


In [8]:
import os
import torch
import wandb
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from source.Config import Config
from source.DatasetTranslation import TranslationDataset
from source.Model import Transformer
from source.Train import train_model, _encode_sentence

config = Config()
config.num_epochs = EFFECTIVE_NUM_EPOCHS
config.batch_size = EFFECTIVE_BATCH_SIZE
config.max_seq_length = MAX_SEQ_LENGTH
config.learning_rate = EFFECTIVE_LEARNING_RATE
config.patience = PATIENCE
config.warmup_steps = WARMUP_STEPS
config.bleu_eval_batches = EFFECTIVE_BLEU_EVAL_BATCHES
config.wandb_project = os.getenv("WANDB_PROJECT", "english-spanish-translator")
config.wandb_entity = os.getenv("WANDB_ENTITY") or None
config.wandb_mode = os.getenv("WANDB_MODE", "offline")

if os.getenv("WANDB_API_KEY"):
    wandb.login(key=os.getenv("WANDB_API_KEY"))

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

os.makedirs(config.tokenizer_path, exist_ok=True)

tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
tokenizer.add_special_tokens(
    {
        "additional_special_tokens": [
            config.pad_token,
            config.unk_token,
            config.sos_token,
            config.eos_token,
        ]
    }
)

config.vocab_size = len(tokenizer)
config.pad_token_id = tokenizer.convert_tokens_to_ids(config.pad_token)
config.unk_token_id = tokenizer.convert_tokens_to_ids(config.unk_token)
config.sos_token_id = tokenizer.convert_tokens_to_ids(config.sos_token)
config.eos_token_id = tokenizer.convert_tokens_to_ids(config.eos_token)

tokenizer.save_pretrained(config.tokenizer_path)

train_dataset_full = TranslationDataset(
    csv_file=config.train_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
split_generator = torch.Generator().manual_seed(config.seed)
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=split_generator,
)

num_workers = NUM_WORKERS if torch.cuda.is_available() else 0
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

model = Transformer(config).to(config.device)
train_model(model, train_dataloader, val_dataloader, config, tokenizer)

checkpoint = torch.load(config.model_save_path, map_location=config.device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Training complete. Best checkpoint:", config.model_save_path)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: relixmatrix (relixmatrix-texas-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret 

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

Validation Epoch 30: 100%|██████████| 549/549 [01:03<00:00,  8.65batch/s]

Epoch 30 Validation Loss: 2.5060


Epoch 30 BLEU Score: 0.2786
Sample translations:
  [How are you?] -> [¿ Cómo estás?]
  [Where is the hospital?] -> [¿ Ingres es el hospital?]
  [I need help with my homework.] -> [Necesito ayuda con mi trabajo doméstico.]
No improvement for 1 epoch(s).
LR: 0.000225


bleu_score,▁▃▃▃▃▄▃▅▃▄▃▄▃▃▄▄▅▄▅▄▆▅▄▄▇▅▄▆█▇
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,██████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
bleu_score,0.27857
epoch,30
lr,0.00022
train_loss,2.60653
val_loss,2.50596


Training complete. Best checkpoint: best_model.pth


## 8. Evaluate Model

This cell evaluates the best checkpoint against the test split created in the preprocessing step and writes `bleu_score_distribution.png`.


In [9]:
from source.Evaluate import evaluate_model

test_dataset = TranslationDataset(
    csv_file=config.test_csv,
    tokenizer=tokenizer,
    sequence_length=config.max_seq_length,
    config=config,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

avg_bleu = evaluate_model(
    model,
    test_dataloader,
    config,
    tokenizer,
    max_seq_length=config.max_seq_length,
)

print("Final BLEU:", avg_bleu)


Evaluating: 100%|██████████| 1373/1373 [2:33:37<00:00,  6.71s/batch]


Average BLEU Score (sacrebleu): 31.41
Final BLEU: 0.31406433640102166


## 9. Example Translations

Use the trained checkpoint to translate a few sample English sentences.


In [ ]:
examples = [
    "How are you?",
    "Where is the nearest hospital?",
    "I need help with my homework.",
]

for text in examples:
    encoder_input = _encode_sentence(text, tokenizer, config)
    token_ids = model.generate(encoder_input, config, beam_width=BEAM_WIDTH)
    translation = tokenizer.decode(token_ids, skip_special_tokens=True)
    print(f"{text} -> {translation}")


## 10. Export Artifacts

This cell mounts Google Drive, collects the main training artifacts, zips them, and saves the archive into your Drive automatically.


In [11]:
import os
import shutil
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

drive_output_dir = "/content/drive/MyDrive/Projects/english-spanish-translator"
os.makedirs(drive_output_dir, exist_ok=True)

artifact_paths = [
    "best_model.pth",
    "loss_plot.png",
    "bleu_score_distribution.png",
    "data/train.csv",
    "data/test.csv",
    "data/tokenizer",
]

if os.path.isdir("wandb"):
    artifact_paths.append("wandb")

staging_dir = "/tmp/english_spanish_translator_export"
if os.path.exists(staging_dir):
    shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

for artifact in artifact_paths:
    if not os.path.exists(artifact):
        print("Missing:", artifact)
        continue

    destination = os.path.join(staging_dir, os.path.basename(artifact))
    if os.path.isdir(artifact):
        shutil.copytree(artifact, destination)
    else:
        shutil.copy2(artifact, destination)
    print("Added:", artifact)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_base = os.path.join(drive_output_dir, f"english_spanish_translator_{timestamp}")
archive_path = shutil.make_archive(archive_base, "zip", staging_dir)

print("Saved archive to:", archive_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Added: best_model.pth
Added: loss_plot.png
Added: bleu_score_distribution.png
Added: data/train.csv
Added: data/test.csv
Added: data/tokenizer
Added: wandb
Saved archive to: /content/drive/MyDrive/Projects/english-spanish-translator/english_spanish_translator_20260409_155517.zip
